# ObjectDetectionFromScratch — huấn luyện theo notebook mẫu

Tham khảo thứ tự và cấu hình các cell trong `train_tiny_backbone.ipynb` của `Basic_Object_Decetion-main.zip`. Chạy từng mục. Các lượt huấn luyện bắt đầu từ checkpoint riêng của dự án.

- **Kaggle**: thêm hai ZIP `ObjectDetectionFromScratch_code.zip` và `final_public.zip` dưới dạng Input của notebook. Kết quả nằm trong `/kaggle/working/ObjectDetectionFromScratch_results`. Lưu output của phiên để lấy lại checkpoint.
- **Colab**: đặt ZIP mã nguồn trong MyDrive/ObjectDetectionFromScratch. Dataset lấy từ MyDrive/Final - public TTNT. Kết quả lưu trong MyDrive/ObjectDetectionFromScratch/results.
- Bật GPU trước khi chạy cell train. ResNet34 pretrained cần tải trọng số ImageNet chính thức nếu runtime chưa có bản cache.

Split cố định: train 6.000, validation 1.500, test 1.500 ảnh. Checkpoint của split này lưu trong thư mục con split_6000_1500_1500.

In [ ]:
import json
import subprocess
import sys

# Kiểm tra trong tiến trình Python mới để tránh import dở trong kernel.
environment = subprocess.run(
    [sys.executable, "-c",
     "import json, sympy.logic, torch, torchvision; "
     "print(json.dumps({'torch': torch.__version__, "
     "'torchvision': torchvision.__version__, "
     "'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"],
    capture_output=True, text=True,
)
if environment.returncode:
    print(environment.stderr)
    raise RuntimeError("Kiểm tra môi trường thất bại. Hãy khởi động lại phiên rồi chạy lại cell này.")
runtime = json.loads(environment.stdout)
print(runtime)

In [ ]:
from pathlib import Path
from shutil import copytree
from zipfile import ZipFile
import os

if Path("/kaggle/input").is_dir() and Path("/kaggle/working").is_dir():
    code_root = dataset_root = Path("/kaggle/input")
    work_root = Path("/kaggle/working")
    output_root = work_root / "ObjectDetectionFromScratch_results"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    my_drive = Path("/content/drive/MyDrive")
    code_root = my_drive / "ObjectDetectionFromScratch"
    dataset_root = my_drive / "Final - public TTNT"
    work_root = Path("/content")
    output_root = code_root / "results"

def find_zip(root, name):
    if root.is_file() and root.name == name:
        return root
    matches = sorted(root.rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(f"Cần đúng một {name} trong {root}; tìm thấy {len(matches)}.")
    return matches[0]

def has_dataset(root):
    return all((root / path).is_file() for path in (
        "annotations/train.json", "annotations/val.json", "tools/evaluate_predictions.py",
    )) and all((root / path).is_dir() for path in ("train/images", "val/images"))

project = work_root / "ObjectDetectionFromScratch"
project.mkdir(exist_ok=True)
with ZipFile(find_zip(code_root, "ObjectDetectionFromScratch_code.zip")) as archive:
    archive.extractall(project)

local_data = project / "public"
if not (project / ".dataset_ready").is_file() or not has_dataset(local_data):
    unpacked = [path for path in (dataset_root, dataset_root / "public") if has_dataset(path)]
    if unpacked:
        copytree(unpacked[0], local_data, dirs_exist_ok=True)
        print("Dataset:", unpacked[0])
    else:
        dataset_zip = find_zip(dataset_root, "final_public.zip")
        with ZipFile(dataset_zip) as archive:
            archive.extractall(project)
        print("Dataset:", dataset_zip)
    if not has_dataset(local_data):
        raise FileNotFoundError(f"Dataset cần có annotations, train/images và val/images trong {local_data}.")
    (project / ".dataset_ready").touch()

output_root = output_root / "split_6000_1500_1500"
output_root.mkdir(parents=True, exist_ok=True)
os.chdir(project)
print("Project:", project)
print("Output:", output_root)

## Kiểm tra dữ liệu, tham số và tests

Các cell dưới đây cũng có thể chạy trên CPU. Đếm tham số dùng khởi tạo ngẫu nhiên vì số tham số không phụ thuộc vào bộ trọng số.

In [ ]:
subprocess.run([sys.executable, "-m", "tools.check_environment"], check=True)

parameter_check = 'from models.detector import RetinaDetector\nfor name, channels in (\n    ("tiny_0.25", 128), ("tiny_0.5", 128), ("tiny_1.0", 256),\n    ("resnet18", 256), ("resnet34", 256), ("resnet50", 256),\n):\n    model = RetinaDetector(backbone=name, fpn_channels=channels, pretrained=False)\n    parts = [sum(p.numel() for p in component.parameters())\n             for component in (model.backbone, model.fpn, model.head)]\n    print(name, "| backbone / FPN / head:", parts, "| total:", sum(parts))\n    del model'
subprocess.run([sys.executable, "-c", parameter_check], check=True)

subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)

In [ ]:
def run_training(run_name, options):
    if not runtime["gpu"]:
        raise RuntimeError("Bật GPU cho notebook trước khi train.")
    run_dir = output_root / run_name
    last = run_dir / "last.pth"
    command = [
        sys.executable, "-u", "train.py",
        "--train_data", "splits/train.json",
        "--val_data", "splits/val.json",
        "--image_dir", "public/train/images",
        "--val_image_dir", "public/val/images",
        "--checkpoint_dir", str(run_dir),
        "--device", "cuda",
        "--num_workers", "16",
        *options.split(),
    ]
    if last.is_file():
        command.extend(["--resume", str(last)])
    subprocess.run(command, check=True)

## 1. Tiny 0.5 — GPU

Train 300 epoch theo số epoch trong tiny_0.5/training_history.json của mẫu. Dùng các tham số còn lại từ cell Tiny 0.5 của mẫu vì lịch sử không ghi đầy đủ lệnh chạy. Kết quả lưu trong history_recipe/tiny_0.5.

In [ ]:
run_training("history_recipe/tiny_0.5", "--backbone tiny_0.5 --fpn_channels 128 --img_size 384 --batch_size 32 --epochs 300 --lr 1e-3")

## 2. Tiny 1.0 — GPU

70 epoch theo số epoch trong tiny_1.0/training_history.json của mẫu; các tham số còn lại lấy từ cell Tiny 1.0, gồm multi-scale và GIoU.

In [ ]:
run_training("history_recipe/tiny_1.0", "--backbone tiny --fpn_channels 256 --img_size 512 --batch_size 16 --epochs 70 --lr 1e-3 --multi_scale --ms_max_size 640 --reg_loss_type giou")

## 3. ResNet34 + EMA — GPU

40 epoch, pretrained ImageNet, multi-scale và CIoU.

In [ ]:
run_training("resnet34_ema", "--backbone resnet34 --img_size 512 --batch_size 8 --epochs 40 --lr 1e-4 --warmup_iters 1000 --multi_scale --ms_max_size 640 --reg_loss_type ciou --ema")

## 4. ResNet34 + EMA + SWA + TTA — GPU

40 epoch, cùng cấu hình mục 3 và bật SWA, TTA khi chấm validation trong lúc train.

In [ ]:
run_training("resnet34_full", "--backbone resnet34 --img_size 512 --batch_size 8 --epochs 40 --lr 1e-4 --warmup_iters 1000 --multi_scale --ms_max_size 640 --reg_loss_type ciou --ema --swa --use_tta")

## 5. Xuất dự đoán và chấm validation

Dùng `resnet34_full/best.pth` và confidence 0.25 như cell cuối mẫu. Cell mẫu xuất JSON không bật TTA. GPU giúp dự đoán nhanh hơn, bước đọc JSON và tính điểm chạy CPU.

In [ ]:
checkpoint = output_root / "resnet34_full/best.pth"
predictions = output_root / "predictions.json"
score = output_root / "score.json"
subprocess.run([
    sys.executable, "predict.py", "--image_dir", "public/val/images",
    "--checkpoint", str(checkpoint), "--output", str(predictions), "--conf_thresh", "0.25",
], check=True)
subprocess.run([
    sys.executable, "public/tools/evaluate_predictions.py",
    "--ground_truth", "splits/val.json",
    "--predictions", str(predictions), "--output", str(score),
], check=True)

## Chấm tập test riêng

Chỉ chạy sau khi đã chốt checkpoint và ngưỡng trên validation.

In [ ]:
test_predictions = output_root / "test_predictions.json"
subprocess.run([
    sys.executable, "predict.py", "--image_dir", "public/train/images",
    "--annotations", "splits/test.json",
    "--checkpoint", str(output_root / "resnet34_full/best.pth"),
    "--output", str(test_predictions), "--conf_thresh", "0.25",
], check=True)
subprocess.run([
    sys.executable, "public/tools/evaluate_predictions.py",
    "--ground_truth", "splits/test.json",
    "--predictions", str(test_predictions), "--output", str(output_root / "test_score.json"),
], check=True)